# Vol Surface Playground

Self-contained notebook for tinkering with:

1. **Raw IV surface** — scattered market quotes → griddata
2. **Cleaned & smoothed IV surface** — liquidity / arb filter → Gaussian smooth → bicubic spline
3. **Dupire local-vol surface** — Gatheral formula on the smoothed IV

All pipeline code is inlined so you can tweak parameters without touching the API.


In [37]:
# ── 0. Imports ──────────────────────────────────────────────────────
from __future__ import annotations

import warnings
warnings.filterwarnings("ignore")

from collections import defaultdict
from dataclasses import dataclass
from datetime import date, datetime
from typing import List, Optional, Tuple

import numpy as np
import yfinance as yf
import plotly.graph_objects as go
from scipy.interpolate import RectBivariateSpline, griddata
from scipy.ndimage import gaussian_filter

## ▸ Tunable parameters

Change these and re-run the cells below to see the effect.


In [ ]:
# ── Ticker ──────────────────────────────────────────────────────────
TICKER = "AAPL"

# ── Cleaning thresholds ─────────────────────────────────────────────
MIN_OPEN_INTEREST = 10       # minimum OI
MIN_BID           = 0.05     # minimum bid
MAX_SPREAD_RATIO  = 0.50     # max (ask-bid)/mid
MIN_MONEYNESS     = 0.98     # strike/spot lower bound
MAX_MONEYNESS     = 1.02     # strike/spot upper bound
MAX_TTM           = 1.0      # maximum time-to-maturity in years

# ── IV surface grid ────────────────────────────────────────────────
IV_N_Y = 80                  # log-moneyness knots
IV_N_T = 50                  # maturity knots
IV_SMOOTH_SIGMA = 1.5        # Gaussian blur before bicubic fit

# ── Dupire local-vol grid ──────────────────────────────────────────
LV_N_STRIKES    = 100
LV_N_MATURITIES = 50
LV_SMOOTH_SIGMA = 2.0        # Gaussian blur on σ_loc grid

# ── Stability guards (Gatheral) ───────────────────────────────────
DENOM_FLOOR  = 0.05
LOC_VAR_MIN  = 0.01 ** 2     # 1% local vol
LOC_VAR_MAX  = 4.00 ** 2     # 400% local vol

# ── Raw surface display grid ──────────────────────────────────────
RAW_N_STRIKES    = 40
RAW_N_MATURITIES = 30

# ── Risk-free rate (flat) ─────────────────────────────────────────
RISK_FREE_RATE = 0.045       # ~US 1Y Treasury

print("Parameters loaded ✓")

Parameters loaded ✓


---

## Stage 1 — Fetch option chain


In [39]:
# ── OptionQuote dataclass ───────────────────────────────────────────

@dataclass(frozen=True, slots=True)
class OptionQuote:
    ticker: str
    option_type: str        # "call" or "put"
    expiry: date
    ttm: float              # time-to-maturity in years
    strike: float
    bid: float
    ask: float
    mid: float
    last: float
    open_interest: int
    volume: int
    implied_volatility: Optional[float]


def _ttm(expiry: date) -> float:
    return max((expiry - date.today()).days, 0) / 365.0


def _mid(bid: float, ask: float, last: float) -> float:
    if bid > 0 and ask > 0:
        return (bid + ask) / 2.0
    return last


def _parse_chain_df(df, option_type: str, expiry: date, ticker: str) -> List[OptionQuote]:
    ttm = _ttm(expiry)
    if ttm <= 0:
        return []
    quotes = []
    for row in df.itertuples(index=False):
        try:
            strike = float(getattr(row, 'strike', 0))
            if strike <= 0:
                continue
            bid   = float(getattr(row, 'bid',  0) or 0)
            ask   = float(getattr(row, 'ask',  0) or 0)
            last  = float(getattr(row, 'lastPrice', 0) or 0)
            oi    = int(getattr(row, 'openInterest',  0) or 0)
            vol   = int(getattr(row, 'volume',        0) or 0)
            iv_raw = getattr(row, 'impliedVolatility', None)
            iv = float(iv_raw) if (iv_raw is not None and iv_raw == iv_raw and float(iv_raw) > 0) else None
            quotes.append(OptionQuote(
                ticker=ticker, option_type=option_type, expiry=expiry, ttm=ttm,
                strike=strike, bid=bid, ask=ask, mid=_mid(bid, ask, last),
                last=last, open_interest=oi, volume=vol, implied_volatility=iv,
            ))
        except (ValueError, TypeError, AttributeError):
            continue
    return quotes


def fetch_option_chain(ticker: str) -> List[OptionQuote]:
    """Fetch the full option chain for `ticker` from yfinance."""
    tick = yf.Ticker(ticker)
    expirations = tick.options
    if not expirations:
        print(f"⚠ No options for {ticker}")
        return []
    quotes = []
    for exp_str in expirations:
        try:
            expiry = datetime.strptime(exp_str, '%Y-%m-%d').date()
        except ValueError:
            continue
        if _ttm(expiry) <= 0:
            continue
        try:
            chain = tick.option_chain(exp_str)
        except Exception:
            continue
        if chain.calls is not None and not chain.calls.empty:
            quotes.extend(_parse_chain_df(chain.calls, 'call', expiry, ticker))
        if chain.puts is not None and not chain.puts.empty:
            quotes.extend(_parse_chain_df(chain.puts, 'put', expiry, ticker))
    print(f"Fetched {len(quotes)} quotes across {len(expirations)} expirations")
    return quotes

In [40]:
# ── Fetch live data ────────────────────────────────────────────────
tick = yf.Ticker(TICKER)
spot = tick.fast_info.get("lastPrice", tick.fast_info.get("previousClose", 0))
dividend = tick.fast_info.get("dividend_yield", 0.0) or 0.0

raw_quotes = fetch_option_chain(TICKER)

print(f"Spot = {spot:.2f}  |  Dividend yield = {dividend:.4f}  |  Rate = {RISK_FREE_RATE}")
print(f"Unique expiries: {len(set(q.expiry for q in raw_quotes))}")
print(f"Unique strikes:  {len(set(q.strike for q in raw_quotes))}")

Fetched 2387 quotes across 23 expirations
Spot = 273.29  |  Dividend yield = 0.0000  |  Rate = 0.045
Unique expiries: 23
Unique strikes:  115


---

## Stage 2 — Clean option chain


In [ ]:

# ── Cleaning helpers ────────────────────────────────────────────────

@dataclass
class CleaningStats:
    raw_count: int = 0
    after_ttm: int = 0
    after_liquidity: int = 0
    after_moneyness: int = 0
    after_calendar_arbitrage: int = 0
    after_butterfly_arbitrage: int = 0
    final_count: int = 0

    def __str__(self):
        return (
            f"raw={self.raw_count} → "
            f"ttm={self.after_ttm} → "
            f"liquidity={self.after_liquidity} → "
            f"moneyness={self.after_moneyness} → "
            f"calendar_arb={self.after_calendar_arbitrage} → "
            f"butterfly_arb={self.after_butterfly_arbitrage}"
        )


def _liquidity_ok(q, min_oi, min_bid, max_spread_ratio) -> str | None:
    if q.open_interest < min_oi:       return "low_oi"
    if q.implied_volatility is None or q.implied_volatility <= 0: return "no_iv"
    if q.mid <= 0:                     return "mid_le0"
    if q.bid < min_bid:                return "low_bid"
    spread = q.ask - q.bid
    if q.mid > 0 and spread / q.mid > max_spread_ratio: return "wide_spread"
    return None


def _moneyness_ok(q, spot, lo, hi) -> bool:
    return lo <= q.strike / spot <= hi


def _remove_calendar_arbitrage(quotes):
    buckets = defaultdict(list)
    for q in quotes:
        key = (q.option_type, round(q.strike * 2) / 2)
        buckets[key].append(q)
    clean = []
    for series in buckets.values():
        series.sort(key=lambda q: q.ttm)
        max_w = -1.0
        for q in series:
            w = q.implied_volatility ** 2 * q.ttm
            if w >= max_w:
                max_w = w
                clean.append(q)
    return clean


def _remove_butterfly_arbitrage(quotes):
    calls = [q for q in quotes if q.option_type == 'call']
    puts  = [q for q in quotes if q.option_type == 'put']
    by_expiry = defaultdict(list)
    for q in calls:
        by_expiry[q.expiry].append(q)
    clean_calls = []
    for exp_quotes in by_expiry.values():
        exp_quotes.sort(key=lambda q: q.strike)
        changed = True
        while changed and len(exp_quotes) >= 3:
            changed = False
            keep = [True] * len(exp_quotes)
            for i in range(1, len(exp_quotes) - 1):
                if not (keep[i-1] and keep[i] and keep[i+1]):
                    continue
                c0, c1, c2 = exp_quotes[i-1].mid, exp_quotes[i].mid, exp_quotes[i+1].mid
                if c0 - 2*c1 + c2 < 0:
                    keep[i] = False
                    changed = True
            exp_quotes = [q for q, ok in zip(exp_quotes, keep) if ok]
        clean_calls.extend(exp_quotes)
    return clean_calls + puts


def clean_chain(
    quotes, spot,
    min_open_interest=MIN_OPEN_INTEREST,
    min_bid=MIN_BID,
    max_spread_ratio=MAX_SPREAD_RATIO,
    min_moneyness=MIN_MONEYNESS,
    max_moneyness=MAX_MONEYNESS,
    max_ttm=MAX_TTM,
) -> Tuple[List[OptionQuote], CleaningStats]:
    stats = CleaningStats(raw_count=len(quotes))

    # 0. Maturity cap
    mat = [q for q in quotes if q.ttm <= max_ttm]
    stats.after_ttm = len(mat)
    print(f"  TTM cap:    {len(mat):>5} / {len(quotes)}  (TTM ≤ {max_ttm:.1f}y  dropped {len(quotes)-len(mat)})")

    # 1. Liquidity
    reject = {}
    liq = []
    for q in mat:
        r = _liquidity_ok(q, min_open_interest, min_bid, max_spread_ratio)
        if r is None: liq.append(q)
        else: reject[r] = reject.get(r, 0) + 1
    stats.after_liquidity = len(liq)
    print(f"  Liquidity:  {len(liq):>5} / {len(mat)}  rejected: {reject}")

    # 2. Moneyness
    mon = [q for q in liq if _moneyness_ok(q, spot, min_moneyness, max_moneyness)]
    stats.after_moneyness = len(mon)
    print(f"  Moneyness:  {len(mon):>5} / {len(liq)}  (dropped {len(liq)-len(mon)})")

    # 3. Calendar arb
    cal = _remove_calendar_arbitrage(mon)
    stats.after_calendar_arbitrage = len(cal)
    print(f"  Cal arb:    {len(cal):>5} / {len(mon)}  (dropped {len(mon)-len(cal)})")

    # 4. Butterfly arb
    but = _remove_butterfly_arbitrage(cal)
    stats.after_butterfly_arbitrage = len(but)
    stats.final_count = len(but)
    print(f"  Butterfly:  {len(but):>5} / {len(cal)}  (dropped {len(cal)-len(but)})")
    print(f"  Pipeline:   {stats}")

    return but, stats


In [41]:
# ── Run cleaning ──────────────────────────────────────────────────
clean_quotes, cleaning_stats = clean_chain(raw_quotes, spot)
print(f"\n✓ {cleaning_stats.final_count} clean quotes ready for surface fitting")

  Liquidity:   1791 / 2387  rejected: {'low_oi': 286, 'low_bid': 292, 'wide_spread': 18}
  Moneyness:     96 / 1791  (dropped 1695)
  Cal arb:       96 / 96  (dropped 0)
  Butterfly:     96 / 96  (dropped 0)
  Pipeline:   raw=2387 → liquidity=1791 → moneyness=96 → calendar_arb=96 → butterfly_arb=96

✓ 96 clean quotes ready for surface fitting


---

## Stage 3 — Raw IV surface (market quotes, no smoothing)


In [48]:
# ── Raw IV surface via griddata (on cleaned quotes) ─────────────────

def plot_raw_iv_surface(quotes, spot, rate=RISK_FREE_RATE, dividend=0.0,
                        n_strikes=RAW_N_STRIKES, n_maturities=RAW_N_MATURITIES):
    """Quick scatter → linear griddata → Plotly surface."""
    strikes = np.array([q.strike for q in quotes if q.implied_volatility])
    ttms    = np.array([q.ttm    for q in quotes if q.implied_volatility])
    ivs     = np.array([q.implied_volatility for q in quotes if q.implied_volatility])

    K_grid = np.linspace(strikes.min(), strikes.max(), n_strikes)
    T_grid = np.linspace(ttms.min(), ttms.max(), n_maturities)
    KK, TT = np.meshgrid(K_grid, T_grid, indexing='ij')

    iv_grid = griddata(
        points=np.column_stack([strikes, ttms]),
        values=ivs,
        xi=(KK, TT),
        method='linear',
        fill_value=np.nan,
    )
    # Fill NaN with nearest
    nan_mask = np.isnan(iv_grid)
    if  nan_mask.any():
        iv_nn = griddata(
            points=np.column_stack([strikes, ttms]),
            values=ivs, xi=(KK, TT), method='nearest',
        )
        iv_grid = np.where(nan_mask, iv_nn, iv_grid)

    fig = go.Figure(data=[go.Surface(
        x=T_grid, y=K_grid, z=iv_grid,
        colorscale='Viridis',
        colorbar=dict(title='IV'),
    )])
    fig.update_layout(
        title=f'Raw IV Surface — {TICKER}',
        scene=dict(xaxis_title='TTM (yrs)', yaxis_title='Strike', zaxis_title='IV'),
        width=900, height=650,
    )
    fig.show()
    return iv_grid, K_grid, T_grid


raw_iv_grid, raw_K, raw_T = plot_raw_iv_surface(raw_quotes, spot, dividend=dividend)

In [46]:
centered_quotes = [quote for quote in raw_quotes if ( spot *  0.95 <quote.strike < spot * 1.05) and quote.ttm < 0.5 ]
print(f"Unique expiries: {len(set(q.expiry for q in centered_quotes))}")
print(f"Unique strikes:  {len(set(q.strike for q in centered_quotes))}")

Unique expiries: 13
Unique strikes:  11


In [47]:
raw_iv_grid, raw_K, raw_T = plot_raw_iv_surface(centered_quotes, spot, dividend=dividend)


---

## Stage 3b — Cleaned & smoothed IV surface (bicubic spline)


In [17]:
# ── IVSurface class ─────────────────────────────────────────────────

class IVSurface:
    """
    Smooth bicubic IV surface in (y, T) = (ln(K/F_T), TTM) space.
    """
    def __init__(self, spline, spot, rate, dividend, y_min, y_max, t_min, t_max):
        self._spline = spline
        self.spot = spot
        self.rate = rate
        self.dividend = dividend
        self.y_min, self.y_max = y_min, y_max
        self.t_min, self.t_max = t_min, t_max

    def _forward(self, T):
        return self.spot * np.exp((self.rate - self.dividend) * T)

    def _log_moneyness(self, K, T):
        return np.log(K / self._forward(T))

    def _clip(self, y, T):
        return float(np.clip(y, self.y_min, self.y_max)), float(np.clip(T, self.t_min, self.t_max))

    def sigma(self, K, T):
        """σ(K, T)"""
        y = self._log_moneyness(K, T)
        y_c, T_c = self._clip(y, T)
        return max(float(self._spline(y_c, T_c, grid=False)), 1e-6)

    def __call__(self, K, T):
        return self.sigma(K, T)

    def total_variance(self, K, T):
        """w(K,T) = σ²·T"""
        return self.sigma(K, T) ** 2 * T

    # ── Derivatives of w(y, T) ──────────────────────────────────────
    def dw_dT(self, K, T, dT=1e-4):
        T1 = max(T - dT, self.t_min)
        T2 = min(T + dT, self.t_max)
        return (self.total_variance(K, T2) - self.total_variance(K, T1)) / (T2 - T1)

    def dw_dy(self, K, T):
        y = self._log_moneyness(K, T)
        y_c, T_c = self._clip(y, T)
        sig = self.sigma(K, T)
        dsig_dy = float(self._spline(y_c, T_c, dx=1, grid=False))
        return 2.0 * sig * T * dsig_dy

    def d2w_dy2(self, K, T):
        y = self._log_moneyness(K, T)
        y_c, T_c = self._clip(y, T)
        sig = self.sigma(K, T)
        dsig_dy  = float(self._spline(y_c, T_c, dx=1, grid=False))
        d2sig_dy2 = float(self._spline(y_c, T_c, dx=2, grid=False))
        return 2.0 * T * (dsig_dy**2 + sig * d2sig_dy2)

In [18]:
# ── Build IV surface ────────────────────────────────────────────────

def build_iv_surface(
    quotes, spot, rate=RISK_FREE_RATE, dividend=0.0,
    n_y=IV_N_Y, n_t=IV_N_T, smooth_sigma=IV_SMOOTH_SIGMA,
) -> IVSurface:
    # Scatter points in (y, T, σ) space
    y_pts, T_pts, sig_pts = [], [], []
    for q in quotes:
        if q.implied_volatility is None or q.ttm <= 0:
            continue
        F = spot * np.exp((rate - dividend) * q.ttm)
        y = np.log(q.strike / F)
        y_pts.append(y); T_pts.append(q.ttm); sig_pts.append(q.implied_volatility)

    assert len(y_pts) >= 16, f"Only {len(y_pts)} IV points — too few to fit"

    y_arr, T_arr, s_arr = np.array(y_pts), np.array(T_pts), np.array(sig_pts)
    y_min, y_max = y_arr.min() - 0.02, y_arr.max() + 0.02
    t_min, t_max = T_arr.min(), T_arr.max()

    # Regular grid
    y_grid = np.linspace(y_min, y_max, n_y)
    T_grid = np.linspace(t_min, t_max, n_t)
    YY, TT = np.meshgrid(y_grid, T_grid, indexing='ij')

    sig_grid = griddata(
        np.column_stack([y_arr, T_arr]), s_arr, (YY, TT), method='linear', fill_value=np.nan,
    )
    nan_mask = np.isnan(sig_grid)
    if nan_mask.any():
        sig_nn = griddata(np.column_stack([y_arr, T_arr]), s_arr, (YY, TT), method='nearest')
        sig_grid = np.where(nan_mask, sig_nn, sig_grid)
    sig_grid = np.maximum(sig_grid, 0.01)

    # Gaussian smooth → kills Runge oscillations before cubic fit
    # sig_grid = gaussian_filter(sig_grid, sigma=smooth_sigma)
    # sig_grid = np.maximum(sig_grid, 0.01)

    spline = RectBivariateSpline(y_grid, T_grid, sig_grid, kx=3, ky=3)

    print(f"IV Surface: {len(y_pts)} scatter pts | y=[{y_min:.3f},{y_max:.3f}] T=[{t_min:.3f},{t_max:.3f}]")
    print(f"  Grid: {n_y}×{n_t} | Gaussian σ = {smooth_sigma}")

    return IVSurface(spline, spot, rate, dividend, y_min, y_max, t_min, t_max)

In [19]:
# ── Build & plot cleaned IV surface ─────────────────────────────────

iv_surf = build_iv_surface(clean_quotes, spot, dividend=dividend)

# Evaluate on a display grid (strike, TTM)
N_DISP_K, N_DISP_T = 60, 40
F_lo = spot * np.exp((RISK_FREE_RATE - dividend) * iv_surf.t_min)
F_hi = spot * np.exp((RISK_FREE_RATE - dividend) * iv_surf.t_max)
disp_K = np.linspace(F_lo * np.exp(iv_surf.y_min), F_hi * np.exp(iv_surf.y_max), N_DISP_K)
disp_T = np.linspace(iv_surf.t_min, iv_surf.t_max, N_DISP_T)

iv_grid = np.zeros((N_DISP_K, N_DISP_T))
for i, K in enumerate(disp_K):
    for j, T in enumerate(disp_T):
        try:
            iv_grid[i, j] = iv_surf.sigma(K, T)
        except Exception:
            iv_grid[i, j] = np.nan

fig = go.Figure(data=[go.Surface(
    x=disp_T, y=disp_K, z=iv_grid,
    colorscale='Viridis',
    colorbar=dict(title='σ_impl'),
)])
fig.update_layout(
    title=f'Cleaned & Smoothed IV Surface — {TICKER} (Gaussian σ={IV_SMOOTH_SIGMA})',
    scene=dict(xaxis_title='TTM', yaxis_title='Strike', zaxis_title='IV'),
    width=900, height=650,
)
fig.show()

IV Surface: 97 scatter pts | y=[-0.154,0.031] T=[0.003,2.803]
  Grid: 80×50 | Gaussian σ = 1.5


---

## Stage 4 — Dupire local-volatility surface


In [20]:
# ── Gatheral formula ────────────────────────────────────────────────

def gatheral_local_var(iv_surf: IVSurface, K: float, T: float) -> float:
    """σ²_loc(K, T) via the Gatheral numerically-stable Dupire formula."""
    w = iv_surf.total_variance(K, T)
    if w < 1e-8:
        return np.nan

    y = iv_surf._log_moneyness(K, T)
    dw_dT   = iv_surf.dw_dT(K, T)
    dw_dy   = iv_surf.dw_dy(K, T)
    d2w_dy2 = iv_surf.d2w_dy2(K, T)

    if dw_dT < 0:
        dw_dT = max(dw_dT, 1e-6)   # force monotonicity

    denom = (
        1.0
        - (y / w) * dw_dy
        + 0.25 * (-0.25 - 1.0/w + y**2/w**2) * dw_dy**2
        + 0.5 * d2w_dy2
    )

    if abs(denom) < DENOM_FLOOR:
        return np.nan

    lv = dw_dT / denom
    return lv if lv >= 0 else np.nan

In [21]:
# ── Calibrate Dupire ────────────────────────────────────────────────

def calibrate_dupire(
    iv_surf, spot, rate=RISK_FREE_RATE, dividend=0.0,
    n_strikes=LV_N_STRIKES, n_maturities=LV_N_MATURITIES,
    smooth_sigma=LV_SMOOTH_SIGMA,
):
    y_lo, y_hi = iv_surf.y_min, iv_surf.y_max
    t_lo, t_hi = iv_surf.t_min, iv_surf.t_max

    T_grid = np.linspace(t_lo, t_hi, n_maturities)
    T_mid = 0.5 * (t_lo + t_hi)
    F_mid = spot * np.exp((rate - dividend) * T_mid)
    K_lo  = F_mid * np.exp(y_lo)
    K_hi  = F_mid * np.exp(y_hi)
    K_grid = np.linspace(max(K_lo, spot * 0.2), min(K_hi, spot * 3.0), n_strikes)

    loc_var = np.zeros((n_strikes, n_maturities))
    nan_count = 0
    for i, K in enumerate(K_grid):
        for j, T in enumerate(T_grid):
            lv = gatheral_local_var(iv_surf, K, T)
            if np.isnan(lv):
                nan_count += 1
            loc_var[i, j] = lv

    # Fill NaN with nearest-neighbour
    KK, TT = np.meshgrid(K_grid, T_grid, indexing='ij')
    valid = ~np.isnan(loc_var)
    assert valid.sum() >= 4, "Too few valid local-vol cells"
    if not valid.all():
        filled = griddata(
            np.column_stack([KK[valid], TT[valid]]), loc_var[valid],
            np.column_stack([KK.ravel(), TT.ravel()]), method='nearest',
        ).reshape(n_strikes, n_maturities)
        loc_var = np.where(valid, loc_var, filled)

    loc_var = np.clip(loc_var, LOC_VAR_MIN, LOC_VAR_MAX)
    sigma_loc = np.sqrt(loc_var)

    # Smooth
    sigma_loc = gaussian_filter(sigma_loc, sigma=smooth_sigma)
    sigma_loc = np.clip(sigma_loc, np.sqrt(LOC_VAR_MIN), np.sqrt(LOC_VAR_MAX))

    print(f"Dupire: {n_strikes}×{n_maturities} grid | {nan_count} NaN filled | Gaussian σ = {smooth_sigma}")
    print(f"  σ_loc range: [{sigma_loc.min():.4f}, {sigma_loc.max():.4f}]")
    print(f"  K range:     [{K_grid[0]:.1f}, {K_grid[-1]:.1f}]")
    print(f"  T range:     [{t_lo:.3f}, {t_hi:.3f}]")

    return K_grid, T_grid, sigma_loc

In [22]:
# ── Calibrate & plot ────────────────────────────────────────────────

lv_K, lv_T, lv_sigma = calibrate_dupire(iv_surf, spot, dividend=dividend)

fig = go.Figure(data=[go.Surface(
    x=lv_T, y=lv_K, z=lv_sigma,
    colorscale='Hot',
    colorbar=dict(title='σ_loc'),
)])
fig.update_layout(
    title=f'Dupire Local-Vol Surface — {TICKER} (Gaussian σ={LV_SMOOTH_SIGMA})',
    scene=dict(xaxis_title='TTM', yaxis_title='Strike', zaxis_title='Local Vol'),
    width=900, height=650,
)
fig.show()

Dupire: 100×50 grid | 669 NaN filled | Gaussian σ = 2.0
  σ_loc range: [0.0119, 0.6589]
  K range:     [248.5, 298.8]
  T range:     [0.003, 2.803]


---

## Side-by-side comparison


In [24]:
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=2, cols=2,
    specs=[[{'type': 'surface'}, {'type': 'surface'}, {'type': 'surface'}]],
    subplot_titles=['Raw IV', 'Cleaned IV', 'Local Vol'],
    horizontal_spacing=0.02,
)

fig.add_trace(go.Surface(x=raw_T, y=raw_K, z=raw_iv_grid, colorscale='Viridis', showscale=False), row=1, col=1)
fig.add_trace(go.Surface(x=disp_T, y=disp_K, z=iv_grid, colorscale='Viridis', showscale=False), row=1, col=2)
fig.add_trace(go.Surface(x=lv_T,  y=lv_K,  z=lv_sigma,   colorscale='Hot',     showscale=False), row=2, col=1)

fig.update_layout(width=1600, height=550, title=f'{TICKER} — Pipeline Overview')
fig.show()

ValueError: 
The 'specs' argument to make_subplots must be a 2D list of dictionaries with dimensions (2 x 2).
    Received value of type <class 'list'>: [[{'type': 'surface'}, {'type': 'surface'}, {'type': 'surface'}]]

---

## Scratch cells — tinker below


In [ ]:
# Try different smoothing: rebuild surfaces with custom sigma
# iv_surf2 = build_iv_surface(clean_quotes, spot, dividend=dividend, smooth_sigma=0.5)
# lv_K2, lv_T2, lv_sigma2 = calibrate_dupire(iv_surf2, spot, dividend=dividend, smooth_sigma=1.0)

In [ ]:
# Inspect a single expiry slice of clean quotes
# expiry_of_interest = sorted(set(q.expiry for q in clean_quotes))[3]  # 4th expiry
# slice_q = [q for q in clean_quotes if q.expiry == expiry_of_interest and q.option_type == 'call']
# slice_q.sort(key=lambda q: q.strike)
# for q in slice_q:
#     print(f"  K={q.strike:8.1f}  IV={q.implied_volatility:.4f}  OI={q.open_interest}")